In [1]:
from pymodbus.client import AsyncModbusSerialClient
from pymodbus import FramerType
import logging
import platform
import time
import asyncio
import rich

In [2]:
# --- Modbus Client Setup ---
port = "COM12" if platform.system() == "Windows" else "/dev/ttyUSB0"
client = AsyncModbusSerialClient(
    framer=FramerType.RTU, port=port, baudrate=115200, timeout=0.7
)

if not await client.connect():
    print("Failed to connect Modbus.")
else:
    print(f"Connected on port {port}")

Connected on port COM12


In [4]:
HR = {
    1: {  # SLAVE 1 → motor vertical
        "HOMING":       0,
        "WORK_POS_H":   1,
        "WORK_POS_L":   2,
        "MOVE_CMD":     3,
        "MOVE_TARGET_H":4,
        "MOVE_TARGET_L":5,
        "MOVE_STATUS":  6,
    },
    2: {  # SLAVE 2 → módulo elevador/telescópio (exemplo)
        "HOMING": 10,
        "M2_CMD": 50, "M2_TGT": 51, "M2_STS": 52,
        "M3_CMD": 53, "M3_TGT": 54, "M3_STS": 55,
    },
}

In [5]:
async def read_regs(slave, addr, n=1):
    r = await client.read_holding_registers(address=addr, count=n, device_id=slave)
    return r.registers if (r is not None and not r.isError()) else None

async def write_reg(slave, addr, val):
    if not isinstance(val, list):
        val = [val]
    r = await client.write_registers(address=addr, values = val, device_id=slave)
    return (r is not None and not r.isError())

async def read_holding(slave, addr, count=1):
    r = await client.read_holding_registers(address=addr, count=count, device_id =slave)
    return None if r.isError() else r.registers

async def read_inputs(slave = 4, addr =0, count=1):
    r = await client.read_input_registers(address=addr, count=count, device_id =slave)
    return None if r.isError() else r.registers

async def wait_until(slave, addr, val_ok, tmax=90, poll=0.3):
    t0 = time.time()
    while time.time() - t0 < tmax:
        r = await read_regs(slave, addr, 1)
        if r is not None and r[0] == val_ok:
            return True
        await asyncio.sleep(poll)
    return False


async def read_coils(default_slave, addr, counts=1):
    r = await client.read_coils(address=addr, count=counts, device_id= default_slave)
    return None if r.isError() else r.bits

async def write_coil(default_slave, addr, val):
    r = await client.write_coils(address=addr, values = [val], device_id = default_slave)
    result  = await client.write_coil(0, True, device_id=4)
    return not r.isError()


In [6]:
async def do_homing(slave):
    print("do_homing called: ", slave)
    addr = HR[slave]["HOMING"]
    print(f"\n[SLAVE {slave}] Iniciar homing (HR{addr})")
    await write_reg(slave, addr, 1)
    ok = await wait_until(slave, addr, 2, tmax=120)
    print("   → concluído" if ok else "   ⚠ timeout")

In [7]:
async def move_motor1(target):
    """
    Movimento do motor vertical em ticks relativos ao home.
    target: signed 32-bit (pode ser negativo, pode ser > 65535).
    """
    orig_target = target
    if target < 0:
        target &= 0xFFFFFFFF  # 2's complement

    high = (target >> 16) & 0xFFFF
    low  = target & 0xFFFF

    print(f"\n[SLAVE 1] M1 -> {orig_target} ticks (high={high}, low={low})")

    # limpar status
    await write_reg(1, HR[1]["MOVE_STATUS"], 0)

    # escrever alvo 32-bit (2x FC06)
    await write_reg(1, HR[1]["MOVE_TARGET_H"], high)
    await write_reg(1, HR[1]["MOVE_TARGET_L"], low)

    # debug: ler de volta
    r = await read_regs(1, HR[1]["MOVE_TARGET_H"], 2)
    print(f"[DEBUG][M1] HR4-5 lidos: {r}")

    # disparar comando
    await write_reg(1, HR[1]["MOVE_CMD"], 1)

    ok = await wait_until(1, HR[1]["MOVE_STATUS"], 2, tmax=120)
    print("   → alvo atingido" if ok else "   ⚠ timeout")

async def move_motor2(target):
    """
    Exemplo de comando 16-bit (antigo) para slave 2.
    Aqui ainda tratamos como 16-bit signed em 1 reg.
    """
    print(f"\n[SLAVE 2] M2 -> {target}")
    await write_reg(2, HR[2]["M2_STS"], 0)

    # Converter signed para unsigned 16-bit (Modbus)
    modbus_val = target & 0xFFFF

    await write_reg(2, HR[2]["M2_TGT"], modbus_val)
    await write_reg(2, HR[2]["M2_CMD"], 1)
    ok = await wait_until(2, HR[2]["M2_STS"], 2, tmax=120)
    print("   → alvo atingido" if ok else "   ⚠ timeout")

async def move_motor3(target):
    """
    Outro exemplo 16-bit direto para slave 2.
    """
    print(f"\n[SLAVE 2] M3 -> {target}")
    await write_reg(2, HR[2]["M3_STS"], 0)
    await write_reg(2, HR[2]["M3_TGT"], target & 0xFFFF)
    await write_reg(2, HR[2]["M3_CMD"], 1)
    ok = await wait_until(2, HR[2]["M3_STS"], 2, tmax=120)
    print("   → alvo atingido" if ok else "   ⚠ timeout")


In [44]:
await do_homing(1)

do_homing called:  1

[SLAVE 1] Iniciar homing (HR0)
   → concluído


In [25]:
await move_motor1(0)


[SLAVE 1] M1 -> 0 ticks (high=0, low=0)
[DEBUG][M1] HR4-5 lidos: [0, 0]
   → alvo atingido


In [45]:
for i in range(1, 3):
    await move_motor1(50000)
    await asyncio.sleep(2)
    await move_motor1(0)
    await asyncio.sleep(5)


[SLAVE 1] M1 -> 50000 ticks (high=0, low=50000)
[DEBUG][M1] HR4-5 lidos: [0, 50000]
   → alvo atingido

[SLAVE 1] M1 -> 0 ticks (high=0, low=0)
[DEBUG][M1] HR4-5 lidos: [0, 0]


CancelledError: 

In [19]:
await do_homing(2)

do_homing called:  2

[SLAVE 2] Iniciar homing (HR10)
   → concluído


In [79]:
# A complete one cycle Right tested well 
await do_homing(1)
await move_motor1(5000)
await move_motor3(19800)
await move_motor2(-15000)
await move_motor1(1500)
await move_motor2(0) 
await move_motor3(0)
#-------------------
#move_motor1(36000)
#move_motor2(29000)
#gripping

do_homing called:  1

[SLAVE 1] Iniciar homing (HR0)
   → concluído

[SLAVE 1] M1 -> 5000 ticks (high=0, low=5000)
[DEBUG][M1] HR4-5 lidos: [0, 5000]
   → alvo atingido

[SLAVE 2] M3 -> 19800
   → alvo atingido

[SLAVE 2] M2 -> -15000
   → alvo atingido

[SLAVE 1] M1 -> 1500 ticks (high=0, low=1500)
[DEBUG][M1] HR4-5 lidos: [0, 1500]
   → alvo atingido

[SLAVE 2] M2 -> 0
   → alvo atingido

[SLAVE 2] M3 -> 0
   → alvo atingido


In [ ]:
await move_motor1(-500)
await do_homing(1)
await move_motor1(1500)
await move_motor3(19800)
await move_motor2(-15000)
await move_motor1(5000) 
await move_motor2(0)
await move_motor3(0)
await move_motor1(0)


[SLAVE 1] M1 -> -500 ticks (high=65535, low=65036)
[DEBUG][M1] HR4-5 lidos: [65535, 65036]
   → alvo atingido
do_homing called:  1

[SLAVE 1] Iniciar homing (HR0)
   → concluído

[SLAVE 1] M1 -> 1500 ticks (high=0, low=1500)
[DEBUG][M1] HR4-5 lidos: [0, 1500]
   → alvo atingido

[SLAVE 2] M3 -> 19800
   → alvo atingido

[SLAVE 2] M2 -> -15000
   → alvo atingido

[SLAVE 1] M1 -> 5000 ticks (high=0, low=5000)
[DEBUG][M1] HR4-5 lidos: [0, 5000]
   → alvo atingido

[SLAVE 2] M2 -> 0
   → alvo atingido

[SLAVE 2] M3 -> 0
   → alvo atingido

[SLAVE 1] M1 -> 0 ticks (high=0, low=0)
[DEBUG][M1] HR4-5 lidos: [0, 0]
   → alvo atingido


In [ ]:
async def despensing_action(SLAVE_ID = 4, HR_CMD = 0, IR_WEIGHT = 0, COIL_TARE = 0, TARGET_GRAMS = 30, TIMEOUT_S = 180):
    print(f"⏱️ Requesting TARE (coil[0]=1)...")
    if not await write_coil(SLAVE_ID, COIL_TARE, True):
        print(f"❌ Failed to set coil[0]=1 (tare request)")

    # wait until slave clears coil[0] → tare complete
    t0 = time.time()
    while True:
        bits = await read_coils(SLAVE_ID, COIL_TARE, 1)
        if bits and not bits[0]:
            print(f"✅ TARE done (coil cleared).")
            break
        if time.time() - t0 > 3.0:
            print(f"⚠️ Timeout waiting for TARE to finish.")
            break
        await asyncio.sleep(0.1)

    # optional: check weight (should be ~0)
    ri = await read_regs(slave= SLAVE_ID, addr= IR_WEIGHT, n= 1)
    w0 = ri[0] if ri else None
    print(f"Weight after TARE: {w0} g\n")

    # --- Start dispense ---
    print(f"▶ Starting dispense: target {TARGET_GRAMS} g")
    if not await write_reg(slave = SLAVE_ID, addr= HR_CMD, val= [1, TARGET_GRAMS]):
        print(f"❌ Failed to write CMD/TARGET.")

    print(f"✅ CMD=1, TARGET={TARGET_GRAMS} written.\n")

    # --- Monitor loop ---
    t0 = time.time()
    stable_err = 0
    last_w = None

    while True:
        hr = await read_holding(slave= SLAVE_ID, addr=HR_CMD, count= 1)
        ir = await read_inputs(slave = SLAVE_ID, addr = IR_WEIGHT, count= 1)
        
        cmd = hr[0] if hr else None
        w   = ir[0] if ir else None

        if cmd is None or w is None:
            stable_err += 1
            if stable_err > 5:
                print(f"⚠️ Communication lost repeatedly – aborting.")
                break
            await asyncio.sleep(0.3)
            continue
        stable_err = 0

        if w != last_w:
            print(f"   CMD={cmd}, Weight={w} g", end="\r", flush=True)
            last_w = w

        if cmd == 0:
            print(f"\n✅ Dispense complete (CMD=0). Final weight: {w} g.")
            break

        if time.time() - t0 > TIMEOUT_S:
            print(f"\n⏰ Timeout after {TIMEOUT_S}s, forcing CMD=0.")
            await write_reg(slave= SLAVE_ID, addr= HR_CMD, val= [0])
            break
        
        await asyncio.sleep(0.3)



In [ ]:
await despensing_action(SLAVE_ID = 4, HR_CMD = 0, IR_WEIGHT = 0, COIL_TARE = 0, TARGET_GRAMS = 30, TIMEOUT_S = 180)

C:\Users\tanim\AppData\Local\Temp\ipykernel_14176\1302552542.py:1: RuntimeWarning: coroutine 'despensing_action' was never awaited
  despensing_action(SLAVE_ID = 4, HR_CMD = 0, IR_WEIGHT = 0, COIL_TARE = 0, TARGET_GRAMS = 30, TIMEOUT_S = 180)


<coroutine object despensing_action at 0x000002806C175D20>

In [69]:
await despensing_action(SLAVE_ID = 3, HR_CMD = 0, IR_WEIGHT = 0, COIL_TARE = 0, TARGET_GRAMS = 30, TIMEOUT_S = 180)

⏱️ Requesting TARE (coil[0]=1)...
✅ TARE done (coil cleared).
Weight after TARE: 0 g

▶ Starting dispense: target 30 g
✅ CMD=1, TARGET=30 written.

   CMD=1, Weight=47 g
✅ Dispense complete (CMD=0). Final weight: 47 g.


In [ ]:
tasks = [
    asyncio.create_task(
        despensing_action(
            SLAVE_ID=4, HR_CMD=0, IR_WEIGHT=0, COIL_TARE=0, TARGET_GRAMS=30, TIMEOUT_S=180
        )
    ),
    asyncio.create_task(
        despensing_action(
            SLAVE_ID=3, HR_CMD=0, IR_WEIGHT=0, COIL_TARE=0, TARGET_GRAMS=30, TIMEOUT_S=180
        )
    ),
]
results = await asyncio.gather(*tasks)
print(results)

⏱️ Requesting TARE (coil[0]=1)...
⏱️ Requesting TARE (coil[0]=1)...
✅ TARE done (coil cleared).
Weight after TARE: 0 g

▶ Starting dispense: target 30 g
✅ CMD=1, TARGET=30 written.

✅ TARE done (coil cleared).
Weight after TARE: 0 g

▶ Starting dispense: target 30 g
✅ CMD=1, TARGET=30 written.

   CMD=1, Weight=46 g
✅ Dispense complete (CMD=0). Final weight: 46 g.
   CMD=1, Weight=53 g
✅ Dispense complete (CMD=0). Final weight: 53 g.
